# FedMI Playground

Interactive environment for experimenting with FedMI components on **Google Colab** or **Kaggle**.

---

## 1 · Setup

In [ ]:
import os
REPO_URL = "https://github.com/ha405/FedMI.git"
BRANCH = "cvpr"

if not os.path.isdir("FedMI"):
    get_ipython().system(f"git clone -b {BRANCH} {REPO_URL}")
else:
    # Update repo if already exists to get latest fixes
    get_ipython().system(f"cd FedMI && git pull origin {BRANCH}")

os.chdir("FedMI")

from fedmi.env import setup, print_info
setup()
print_info()

## 2 · Configuration

In [ ]:
from fedmi.env import patch_config
from fedmi.config import ExperimentConfig

cfg = ExperimentConfig()
patch_config(cfg)

print(f"Experiment ready — Device: {cfg.device}")

## 3 · Dataset Playground

Load data and visualize samples.

In [ ]:
import torch
from fedmi.dataset import get_dataset, get_test_dataloader
import matplotlib.pyplot as plt

train_set, test_set = get_dataset(cfg)
loader = get_test_dataloader(test_set, cfg)

images, labels = next(iter(loader))
print(f"Batch shape: {images.shape}")

plt.figure(figsize=(10, 2))
for i in range(5):
    plt.subplot(1, 5, i+1)
    plt.imshow(images[i].permute(1, 2, 0).squeeze(), cmap='gray')
    plt.title(f"Label: {labels[i].item()}")
    plt.axis('off')
plt.show()

## 4 · Model Playground

Instantiate a model and run a forward pass.

In [ ]:
from fedmi.models import get_model

model = get_model(cfg).to(cfg.device)
outputs = model(images.to(cfg.device))
print(f"Output shape: {outputs.shape}")

## 5 · Utilities

List experiments or view logs.

In [ ]:
from fedmi.env import list_experiments, show_log

list_experiments()

# Example: show_log('checkpoints/iid_experiment/logs/training_log.txt')

## 6 · Download Results

In [ ]:
import shutil
from IPython.display import FileLink

if os.path.exists(cfg.output_dir):
    output_zip = f"{os.path.basename(cfg.output_dir)}.zip"
    shutil.make_archive(output_zip.replace('.zip', ''), 'zip', cfg.output_dir)

    print(f"Results zipped to {output_zip}")
    display(FileLink(output_zip))
else:
    print("⚠ No output directory found to zip.")